<a href="https://colab.research.google.com/github/DecioVdA/ProveComm/blob/main/TEST_RandomForest_Traffico.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [159]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.express as px
import requests
from io import BytesIO
import random
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, precision_recall_curve
rd_seed = 42  # fissiamo un random seed per riproducibilità e replicabilità
np.random.seed(rd_seed)
random.seed(rd_seed)

rd_seed = 42


In [172]:

url = "https://github.com/DecioVdA/ProveComm/raw/refs/heads/main/DataSetAddestramento.xlsx"
response = requests.get(url)
df = pd.read_excel(BytesIO(response.content))


In [167]:
df.head()

,Data,GiornoSettimana,Scolastiche_VDA,Scolastiche_FRA,Scolastiche_CH,Festivo_ITA,Festivo_FRA,Festivo_CH,Dispo_TU,Dispo_Fre,TransitiFRA
0,2024-01-01,Lunedi,1,ABC,nullo,FE,FE,FE,1.0,1.0,2274
1,2024-01-02,Martedi,1,ABC,nullo,nullo,nullo,nullo,1.0,1.0,3160
2,2024-01-03,Mercoledi,1,ABC,nullo,nullo,nullo,nullo,1.0,1.0,2667
3,2024-01-04,Giovedi,1,ABC,nullo,nullo,nullo,nullo,1.0,1.0,2633
4,2024-01-05,Venerdi,1,ABC,nullo,nullo,nullo,nullo,1.0,1.0,2608


In [173]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 366 entries, 0 to 365
Data columns (total 11 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   Data             366 non-null    datetime64[ns]
 1   GiornoSettimana  366 non-null    object        
 2   Scolastiche_VDA  366 non-null    int64         
 3   Scolastiche_FRA  366 non-null    object        
 4   Scolastiche_CH   366 non-null    object        
 5   Festivo_ITA      366 non-null    object        
 6   Festivo_FRA      366 non-null    object        
 7   Festivo_CH       366 non-null    object        
 8   Dispo_TU         366 non-null    float64       
 9   Dispo_Fre        366 non-null    float64       
 10  TransitiFRA      366 non-null    int64         
dtypes: datetime64[ns](1), float64(2), int64(2), object(6)
memory usage: 31.6+ KB


In [174]:
df['Data'] = pd.to_datetime(df['Data'])
df['Day'] = df['Data'].dt.day # giorno del mese
df['Day_of_week'] = df['Data'].dt.day_of_week
df['Day_of_year'] = df['Data'].dt.day_of_year
df['Month'] = df['Data'].dt.month


In [175]:
df = df.drop(['Data'], axis = 1)
df = df.drop(['Day'], axis = 1)
df = df.drop(['Day_of_year'], axis = 1)
df = df.drop(['Day_of_week'], axis = 1)
df['Month'] = df['Month'].astype(str)
df['Scolastiche_VDA'] = df['Scolastiche_VDA'].replace({1: 'Si', 0:'No'})
df



,GiornoSettimana,Scolastiche_VDA,Scolastiche_FRA,Scolastiche_CH,Festivo_ITA,Festivo_FRA,Festivo_CH,Dispo_TU,Dispo_Fre,TransitiFRA,Month
0,Lunedi,Si,ABC,nullo,FE,FE,FE,1.0,1.0,2274,1
1,Martedi,Si,ABC,nullo,nullo,nullo,nullo,1.0,1.0,3160,1
2,Mercoledi,Si,ABC,nullo,nullo,nullo,nullo,1.0,1.0,2667,1
3,Giovedi,Si,ABC,nullo,nullo,nullo,nullo,1.0,1.0,2633,1
4,Venerdi,Si,ABC,nullo,nullo,nullo,nullo,1.0,1.0,2608,1
...,...,...,...,...,...,...,...,...,...,...,...
361,Venerdi,Si,ABC,GVZ,nullo,nullo,nullo,1.0,1.0,2789,12
362,Sabato,Si,ABC,GVZ,nullo,nullo,nullo,1.0,1.0,3095,12
363,Domenica,Si,ABC,GVZ,nullo,nullo,nullo,1.0,1.0,2673,12
364,Lunedi,Si,ABC,GVZ,nullo,nullo,nullo,1.0,1.0,2663,12


In [176]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 366 entries, 0 to 365
Data columns (total 11 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   GiornoSettimana  366 non-null    object 
 1   Scolastiche_VDA  366 non-null    object 
 2   Scolastiche_FRA  366 non-null    object 
 3   Scolastiche_CH   366 non-null    object 
 4   Festivo_ITA      366 non-null    object 
 5   Festivo_FRA      366 non-null    object 
 6   Festivo_CH       366 non-null    object 
 7   Dispo_TU         366 non-null    float64
 8   Dispo_Fre        366 non-null    float64
 9   TransitiFRA      366 non-null    int64  
 10  Month            366 non-null    object 
dtypes: float64(2), int64(1), object(8)
memory usage: 31.6+ KB


In [177]:
categoriche = ['Scolastiche_VDA', 'Scolastiche_FRA', 'Scolastiche_CH', 'Festivo_ITA', 'Festivo_FRA', 'Festivo_CH', 'GiornoSettimana', 'Month']
numeriche = list(set(df.columns)- set(df[categoriche]) - {'TransitiFRA'})

In [178]:
list(categoriche)
list(numeriche)

['Dispo_Fre', 'Dispo_TU']

#### Separazione del dataset

In [179]:
X = df.drop(['TransitiFRA'], axis =1)
y = df['TransitiFRA']

In [180]:
X_train, X_test, y_train, y_test = train_test_split(X,y, test_size=0.3, random_state=rd_seed)
X_train.shape[0] + X_test.shape[0] == df.shape[0]

True

In [181]:
encoder = OneHotEncoder().fit(X_train[categoriche])
X_train_cat_enc = encoder.transform(X_train[categoriche]).todense().astype(int)
X_train_cat_enc = pd.DataFrame(X_train_cat_enc, columns = encoder.get_feature_names_out(categoriche), index = X_train.index)
X_test_cat_enc = encoder.transform(X_test[categoriche]).todense().astype(int)
X_test_cat_enc = pd.DataFrame(X_test_cat_enc, columns = encoder.get_feature_names_out(categoriche), index = X_test.index)
X_train_cat_enc

,Scolastiche_VDA_No,Scolastiche_VDA_Si,Scolastiche_FRA_AB,Scolastiche_FRA_ABC,Scolastiche_FRA_AC,Scolastiche_FRA_B,Scolastiche_FRA_C,Scolastiche_FRA_nullo,Scolastiche_CH_G,Scolastiche_CH_GV,...,Month_11,Month_12,Month_2,Month_3,Month_4,Month_5,Month_6,Month_7,Month_8,Month_9
268,1,0,0,0,0,0,0,1,0,0,...,0,0,0,0,0,0,0,0,0,1
231,0,1,0,1,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,1,0
157,0,1,0,0,0,0,0,1,0,0,...,0,0,0,0,0,0,1,0,0,0
19,1,0,0,0,0,0,0,1,0,0,...,0,0,0,0,0,0,0,0,0,0
147,1,0,0,0,0,0,0,1,0,0,...,0,0,0,0,0,1,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
71,1,0,0,0,0,0,0,1,0,0,...,0,0,0,1,0,0,0,0,0,0
106,1,0,0,0,1,0,0,0,0,0,...,0,0,0,0,1,0,0,0,0,0
270,1,0,0,0,0,0,0,1,0,0,...,0,0,0,0,0,0,0,0,0,1
348,1,0,0,0,0,0,0,1,0,0,...,0,1,0,0,0,0,0,0,0,0


In [182]:
list(X_train_cat_enc.columns)

['Scolastiche_VDA_No',
 'Scolastiche_VDA_Si',
 'Scolastiche_FRA_AB',
 'Scolastiche_FRA_ABC',
 'Scolastiche_FRA_AC',
 'Scolastiche_FRA_B',
 'Scolastiche_FRA_C',
 'Scolastiche_FRA_nullo',
 'Scolastiche_CH_G',
 'Scolastiche_CH_GV',
 'Scolastiche_CH_GVZ',
 'Scolastiche_CH_VZ',
 'Scolastiche_CH_Z',
 'Scolastiche_CH_nullo',
 'Festivo_ITA_FE',
 'Festivo_ITA_RI',
 'Festivo_ITA_nullo',
 'Festivo_FRA_FE',
 'Festivo_FRA_nullo',
 'Festivo_CH_FE',
 'Festivo_CH_nullo',
 'GiornoSettimana_Domenica',
 'GiornoSettimana_Giovedi',
 'GiornoSettimana_Lunedi',
 'GiornoSettimana_Martedi',
 'GiornoSettimana_Mercoledi',
 'GiornoSettimana_Sabato',
 'GiornoSettimana_Venerdi',
 'Month_1',
 'Month_10',
 'Month_11',
 'Month_12',
 'Month_2',
 'Month_3',
 'Month_4',
 'Month_5',
 'Month_6',
 'Month_7',
 'Month_8',
 'Month_9']

In [183]:
X_train = pd.concat([X_train[numeriche], X_train_cat_enc], axis=1)
X_test = pd.concat([X_test[numeriche], X_test_cat_enc], axis=1)
X_train

,Dispo_Fre,Dispo_TU,Scolastiche_VDA_No,Scolastiche_VDA_Si,Scolastiche_FRA_AB,Scolastiche_FRA_ABC,Scolastiche_FRA_AC,Scolastiche_FRA_B,Scolastiche_FRA_C,Scolastiche_FRA_nullo,...,Month_11,Month_12,Month_2,Month_3,Month_4,Month_5,Month_6,Month_7,Month_8,Month_9
268,1.000000,0.0,1,0,0,0,0,0,0,1,...,0,0,0,0,0,0,0,0,0,1
231,1.000000,1.0,0,1,0,1,0,0,0,0,...,0,0,0,0,0,0,0,0,1,0
157,1.000000,1.0,0,1,0,0,0,0,0,1,...,0,0,0,0,0,0,1,0,0,0
19,1.000000,1.0,1,0,0,0,0,0,0,1,...,0,0,0,0,0,0,0,0,0,0
147,1.000000,1.0,1,0,0,0,0,0,0,1,...,0,0,0,0,0,1,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
71,0.958333,1.0,1,0,0,0,0,0,0,1,...,0,0,0,1,0,0,0,0,0,0
106,1.000000,1.0,1,0,0,0,1,0,0,0,...,0,0,0,0,1,0,0,0,0,0
270,1.000000,0.0,1,0,0,0,0,0,0,1,...,0,0,0,0,0,0,0,0,0,1
348,1.000000,0.0,1,0,0,0,0,0,0,1,...,0,1,0,0,0,0,0,0,0,0


In [184]:
model = RandomForestClassifier(random_state = rd_seed)
model.fit(X_train, y_train)

RandomForestClassifier(random_state=42)

In [185]:
y_pred_train = model.predict(X_train)
y_pred_test = model.predict(X_test)

In [186]:
print('Prestazione sul training set: {0:0.2%}'. format(accuracy_score(y_train, y_pred_train)))
print('Prestazione sul test set: {0:0.2%}'. format(accuracy_score(y_test, y_pred_test)))

Prestazione sul training set: 85.94%
Prestazione sul test set: 27.27%


In [188]:
from sklearn.tree import DecisionTreeClassifier, plot_tree

In [189]:
gini = DecisionTreeClassifier(criterion = 'gini', max_depth = 4, random_state = rd_seed)
modello = gini.fit(X_train, y_train)

In [190]:
modello

DecisionTreeClassifier(max_depth=4, random_state=42)

In [191]:
y_pred_trainDT = model.predict(X_train)  # test del modello sui dati di training
y_pred_testDT = model.predict(X_test)  # test del modello sui dati di test

In [192]:
print('Prestazione sul training set: {0:0.2%}'. format(accuracy_score(y_train, y_pred_trainDT)))
print('Prestazione sul test set: {0:0.2%}'. format(accuracy_score(y_test, y_pred_testDT)))

Prestazione sul training set: 85.94%
Prestazione sul test set: 27.27%


#### Eliminazioni di variabili non descrittive

Dispo_TU è fortemente legata alla target feature:
In pratica sì, è corretto, ma con una precisazione importante:
non è tanto che “influenzano troppo”, ma che sono deterministici e quindi non contengono informazione utile per l’apprendimento del modello.

In [196]:
url = "https://github.com/DecioVdA/ProveComm/raw/refs/heads/main/DataSetAddestramento.xlsx"
response = requests.get(url)
df = pd.read_excel(BytesIO(response.content))

df_filtered = df[df["Dispo_TU"] > 0].copy()

In [197]:
df_filtered['Data'] = pd.to_datetime(df_filtered['Data'])
df_filtered['Day'] = df_filtered['Data'].dt.day # giorno del mese
df_filtered['Day_of_week'] = df_filtered['Data'].dt.day_of_week
df_filtered['Day_of_year'] = df_filtered['Data'].dt.day_of_year
df_filtered['Month'] = df_filtered['Data'].dt.month
df_filtered = df_filtered.drop(['Data'], axis = 1)
df_filtered = df_filtered.drop(['Day'], axis = 1)
df_filtered = df_filtered.drop(['Day_of_year'], axis = 1)
df_filtered = df_filtered.drop(['Day_of_week'], axis = 1)
df_filtered['Month'] = df_filtered['Month'].astype(str)
df_filtered['Scolastiche_VDA'] = df_filtered['Scolastiche_VDA'].replace({1: 'Si', 0:'No'})
df_filtered

,GiornoSettimana,Scolastiche_VDA,Scolastiche_FRA,Scolastiche_CH,Festivo_ITA,Festivo_FRA,Festivo_CH,Dispo_TU,Dispo_Fre,TransitiFRA,Month
0,Lunedi,Si,ABC,nullo,FE,FE,FE,1.0,1.0,2274,1
1,Martedi,Si,ABC,nullo,nullo,nullo,nullo,1.0,1.0,3160,1
2,Mercoledi,Si,ABC,nullo,nullo,nullo,nullo,1.0,1.0,2667,1
3,Giovedi,Si,ABC,nullo,nullo,nullo,nullo,1.0,1.0,2633,1
4,Venerdi,Si,ABC,nullo,nullo,nullo,nullo,1.0,1.0,2608,1
...,...,...,...,...,...,...,...,...,...,...,...
361,Venerdi,Si,ABC,GVZ,nullo,nullo,nullo,1.0,1.0,2789,12
362,Sabato,Si,ABC,GVZ,nullo,nullo,nullo,1.0,1.0,3095,12
363,Domenica,Si,ABC,GVZ,nullo,nullo,nullo,1.0,1.0,2673,12
364,Lunedi,Si,ABC,GVZ,nullo,nullo,nullo,1.0,1.0,2663,12


In [198]:
X = df_filtered.drop(['TransitiFRA'], axis = 1)
y = df_filtered['TransitiFRA']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state = rd_seed)

In [199]:
encoder = OneHotEncoder().fit(X_train[categoriche])
X_train_cat_enc = encoder.transform(X_train[categoriche]).todense().astype(int)
X_train_cat_enc = pd.DataFrame(X_train_cat_enc, columns = encoder.get_feature_names_out(categoriche), index = X_train.index)
X_test_cat_enc = encoder.transform(X_test[categoriche]).todense().astype(int)
X_test_cat_enc = pd.DataFrame(X_test_cat_enc, columns = encoder.get_feature_names_out(categoriche), index = X_test.index)
X_train_cat_enc

,Scolastiche_VDA_No,Scolastiche_VDA_Si,Scolastiche_FRA_AB,Scolastiche_FRA_ABC,Scolastiche_FRA_AC,Scolastiche_FRA_B,Scolastiche_FRA_C,Scolastiche_FRA_nullo,Scolastiche_CH_G,Scolastiche_CH_GV,...,Month_1,Month_12,Month_2,Month_3,Month_4,Month_5,Month_6,Month_7,Month_8,Month_9
124,1,0,0,0,0,1,0,0,0,0,...,0,0,0,0,0,1,0,0,0,0
164,0,1,0,0,0,0,0,1,0,0,...,0,0,0,0,0,0,1,0,0,0
245,0,1,0,0,0,0,0,1,0,0,...,0,0,0,0,0,0,0,0,0,1
227,0,1,0,1,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,1,0
29,0,1,0,0,0,0,0,1,0,0,...,1,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
106,1,0,0,0,1,0,0,0,0,0,...,0,0,0,0,1,0,0,0,0,0
14,1,0,0,0,0,0,0,1,0,0,...,1,0,0,0,0,0,0,0,0,0
92,1,0,0,0,0,0,0,1,0,1,...,0,0,0,0,1,0,0,0,0,0
180,0,1,0,0,0,0,0,1,0,1,...,0,0,0,0,0,0,1,0,0,0


In [200]:
X_train = pd.concat([X_train[numeriche], X_train_cat_enc], axis=1)
X_test = pd.concat([X_test[numeriche], X_test_cat_enc], axis=1)
X_train

,Dispo_Fre,Dispo_TU,Scolastiche_VDA_No,Scolastiche_VDA_Si,Scolastiche_FRA_AB,Scolastiche_FRA_ABC,Scolastiche_FRA_AC,Scolastiche_FRA_B,Scolastiche_FRA_C,Scolastiche_FRA_nullo,...,Month_1,Month_12,Month_2,Month_3,Month_4,Month_5,Month_6,Month_7,Month_8,Month_9
124,1.0,1.0000,1,0,0,0,0,1,0,0,...,0,0,0,0,0,1,0,0,0,0
164,1.0,1.0000,0,1,0,0,0,0,0,1,...,0,0,0,0,0,0,1,0,0,0
245,1.0,0.8333,0,1,0,0,0,0,0,1,...,0,0,0,0,0,0,0,0,0,1
227,1.0,1.0000,0,1,0,1,0,0,0,0,...,0,0,0,0,0,0,0,0,1,0
29,1.0,1.0000,0,1,0,0,0,0,0,1,...,1,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
106,1.0,1.0000,1,0,0,0,1,0,0,0,...,0,0,0,0,1,0,0,0,0,0
14,1.0,1.0000,1,0,0,0,0,0,0,1,...,1,0,0,0,0,0,0,0,0,0
92,1.0,1.0000,1,0,0,0,0,0,0,1,...,0,0,0,0,1,0,0,0,0,0
180,1.0,1.0000,0,1,0,0,0,0,0,1,...,0,0,0,0,0,0,1,0,0,0


In [201]:
model = RandomForestClassifier(random_state = rd_seed)
model.fit(X_train, y_train)

RandomForestClassifier(random_state=42)

In [202]:
y_pred_train = model.predict(X_train)
y_pred_test = model.predict(X_test)

In [204]:
print('Prestazione sul training set: {0:0.2%}'. format(accuracy_score(y_train, y_pred_train)))
print('Prestazione sul test set: {0:0.2%}'. format(accuracy_score(y_test, y_pred_test)))

Prestazione sul training set: 84.36%
Prestazione sul test set: 0.00%
